# Database and Integration

## Scope: Reproduce Database and Integration from the Supplied Handoff Files

This notebook reproduces **Database and Integration only**. It treats the completed Data Acquisition and Cleaning, Spatial Integration, and External Data Augmentation outputs as fixed inputs, loads them into DuckDB, and validates the resulting database. Re-running upstream acquisition, cleaning, spatial matching or external API retrieval is outside this notebook's reproduction scope.

The following existing files are sufficient to reproduce Database and Integration:

| Required input | Purpose |
| --- | --- |
| `data/processed/chargers_clean.csv` | Data Acquisition and Cleaning's cleaned source records |
| `data/processed/charger_sa4.csv` | Spatial Integration's confirmed spatial results and unmatched status |
| `data/processed/charger_sa4_review.csv` | Spatial Integration's unresolved candidate-region review |
| `data/processed/charger_attributes.csv` | External Data Augmentation's final accepted augmentation output |
| `data/raw/SA4_2026_AUST_SHP_GDA2020.zip` | Complete SA4 boundaries and shapefile components |

External Data Augmentation's raw API caches, candidate-review files and API key are **not required to reproduce Database and Integration**. Database and Integration does not make requests to the external augmentation API.

**Record granularity:** `record_id` identifies a source record, not necessarily a unique physical charging site. Augmentation coverage is reported both by source record and by External Data Augmentation's location definition (latitude and longitude rounded to six decimal places). Unmatched records and upstream quality flags are retained.

### How to reproduce Database and Integration

1. Extract the complete project and install `requirements.txt` in the Python environment used by Jupyter.
2. Keep this notebook in `notebooks/`, with `scripts/build_database.py`, `sql/schema.sql` and the input files in their original project locations.
3. Restart the kernel and run all cells. The first installation of DuckDB's official `spatial` extension requires network access; subsequent runs reuse the local extension cache.
4. Confirm that the build completes, review the validation results, and open the generated `data/final/project.duckdb` using the queries in Section 4.

Running all cells rebuilds the database and exports quality statistics and review lists. To inspect an existing database without rebuilding it, run Section 1 and then Section 4.

The notebook and command-line entry point call the same implementation in `scripts/build_database.py`. The equivalent command from the project root is `python scripts/build_database.py`.


## 1. Environment and Project Paths

Start from the project root or its `notebooks/` directory. No variables from the Data Acquisition and Cleaning, Spatial Integration, and External Data Augmentation notebook kernels are required.

In [ ]:
from pathlib import Path
import sys
import json

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'scripts/build_database.py').is_file()
                     and (p / 'data/processed/chargers_clean.csv').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Please run from the project root or notebooks directory.')
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from build_database import build_database, connect_database
print('Project folder:', PROJECT_ROOT.name)

Project folder: 5339-main


## 2. Database Design and Standalone DDL

- `operators`: standardized operator names as natural primary keys. Original operator labels remain in the base table.
- `charger_records`: all fields supplied by Data Acquisition and Cleaning, plus point geometries. WGS84 is an unconfirmed assumption inherited from Spatial Integration.
- `sa4_regions`: all Australian SA4 categories and EPSG:7844 geometries, including non-spatial categories with missing geometry.
- `charger_sa4`: one result per source record; unmatched regions remain NULL. `spatial_review` stores unresolved candidates separately and does not establish a confirmed assignment.
- `charger_attributes`: a composite primary key of source record, external source and external station ID supports one-to-many matches. `attribute_connectors` separates connector types for querying.
- `build_inputs` / `build_metadata`: input fingerprints, import time and validation results. Import time is not represented as the external retrieval time.

### Design rationale

Operators and regions are stored separately to avoid repeating their shared attributes and large boundary geometries. The base and augmentation tables retain upstream fields for traceability. Spatial Integration's region names are retained as handoff snapshots and checked against the boundary lookup before loading.

A shared external station ID does not imply that source records should be deleted or merged. Connector types are separated for filtering, but connector and power lists are not paired because External Data Augmentation did not supply that relationship. `charger_overview` uses `EXISTS` to indicate augmentation without multiplying source records; `sa4_summary` therefore counts source records rather than physical sites.

The schema diagram and additional design notes are in `docs/database_design.md`. The DDL below can run independently in a new database after installing and loading the `spatial` extension. DDL recreates the structure; the build script loads the data.

In [ ]:
print((PROJECT_ROOT / 'sql/schema.sql').read_text(encoding='utf-8'))

-- Standalone DDL: execute in a new database after INSTALL spatial; LOAD spatial.
-- Names retain upstream fields. Source records must not be labelled unique physical sites.
CREATE TABLE operators (operator_name VARCHAR PRIMARY KEY);
CREATE TABLE sa4_regions (
    SA4_CODE26 VARCHAR PRIMARY KEY,
    SA4_NAME26 VARCHAR NOT NULL,
    STE_CODE26 VARCHAR NOT NULL,
    STE_NAME26 VARCHAR NOT NULL,
    area_sq_km DOUBLE,
    geometry_crs VARCHAR NOT NULL CHECK (geometry_crs = 'EPSG:7844'),
    boundary_version VARCHAR NOT NULL,
    geom GEOMETRY
);
CREATE TABLE charger_records (
    "record_id" VARCHAR PRIMARY KEY,
    "OBJECTID" VARCHAR,
    "Station_name" VARCHAR,
    "Station_address" VARCHAR,
    "Operator" VARCHAR,
    "Number_of_plugs" INTEGER CHECK ("Number_of_plugs" > 0),
    "Charger_Type" VARCHAR,
    "Charger_rating" VARCHAR,
    "Latitude" DOUBLE NOT NULL CHECK (isfinite("Latitude") AND "Latitude" BETWEEN -90 AND 90),
    "Longitude" DOUBLE NOT NULL CHECK (isfinite("Longitude") A

## 3. Build, Load and Validate

The first run requires network access to install the official `spatial` extension; later runs reuse the local cache. The build checks primary and foreign keys, row counts, consistency between Data Acquisition and Cleaning and Spatial Integration, geometry validity, and an independent spatial recheck before replacing the published database. It does not retrieve external API data again.

Input files are read from disk. A temporary database is built first so validation failures do not overwrite the previous successful database. The quality report records exact input hashes and software versions.

The shared build script also records optional upstream provenance information, including `missing_c_external_files`. This field is informational: those External Data Augmentation files are not inputs to Database and Integration, and their absence does not indicate a failed or incomplete Database and Integration reproduction. The Database and Integration acceptance criteria are listed in Section 6.


In [ ]:
quality = build_database(PROJECT_ROOT)
print(json.dumps(quality, indent=2, ensure_ascii=False))

{
  "built_at_utc": "2026-09-22T08:49:22.559326+00:00",
  "python_version": "3.11.2",
  "duckdb_version": "1.5.5",
  "geopandas_version": "1.1.4",
  "base_records": 1958,
  "operators": 46,
  "sa4_regions": 108,
  "sa4_without_geometry": 19,
  "spatial_matched": 1957,
  "spatial_unmatched": 1,
  "spatial_review_rows": 1,
  "spatial_recheck_mismatches": 0,
  "invalid_region_geometries": 0,
  "dc_records": 433,
  "dc_locations_6dp": 430,
  "augmentation_rows": 216,
  "augmented_dc_records": 216,
  "augmented_dc_locations_6dp": 215,
  "dc_record_coverage": 0.49884526558891457,
  "dc_location_coverage": 0.5,
  "coverage_target_met": true,
  "match_distance_over_3km": 17,
  "missing_c_external_files": [
    "ocm_pilot_raw.json",
    "ocm_australia_raw.json",
    "ocm_pilot_candidate_review.csv",
    "ocm_full_candidate_review.csv"
  ],
  "limitations": [
    "record_id identifies a source record, not a unique physical site.",
    "Location coverage uses Python round(latitude/longitude, 6), 

## 4. SQL Queries and Spatial Checks

These cells use read-only connections and close each connection after querying. Counts are labelled as source records to distinguish them from physical charging sites. The spatial comparison explicitly transforms point geometries from the assumed EPSG:4326 to the boundary CRS, EPSG:7844, with longitude/latitude axis order.

In [ ]:
with connect_database(PROJECT_ROOT) as con:
    display(con.execute("SELECT * FROM sa4_summary WHERE source_records > 0 ORDER BY source_records DESC").df())
    display(con.execute("SELECT * FROM spatial_review").df())
    display(con.execute("SELECT * FROM augmentation_review ORDER BY match_distance_m DESC").df())
    display(con.execute("SELECT * FROM reused_external_ids ORDER BY source_record_count DESC").df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,SA4_CODE26,SA4_NAME26,STE_NAME26,source_records,dc_source_records,augmented_source_records
0,118,Sydney - Eastern Suburbs,New South Wales,221,35,7
1,117,Sydney - City and Inner South,New South Wales,139,19,6
2,106,Hunter Valley exc Newcastle,New South Wales,128,9,7
3,101,Capital Region,New South Wales,117,27,23
4,120,Sydney - Inner West,New South Wales,116,17,5
5,103,Central West,New South Wales,113,17,12
6,121,Sydney - North Sydney and Hornsby,New South Wales,109,41,15
7,111,Newcastle and Lake Macquarie,New South Wales,87,13,3
8,114,Southern Highlands and Shoalhaven,New South Wales,76,9,6
9,112,Richmond - Tweed,New South Wales,76,14,7


,review_id,record_id,Station_address,candidate_sa4_code,candidate_sa4_name,candidate_distance_m,review_status,charger_crs_status,distance_crs
0,1,ev_cba86a64f9e7cfd83617245c590c47c767ecd3439cb...,1 Sandy Bay Rd\nClontarf NSW 2093\nAustralia,122,Sydney - Northern Beaches,1.765063,unresolved,Assumed WGS84; pending confirmation,EPSG:32756


,record_id,Station_address,external_station_id,external_station_title,external_address,match_distance_m,match_score,match_method,match_decision,external_last_verified
0,ev_6fddf67cd79ce70f1669da6b18638c91bb7b066b4cd...,"116 Liverpool St, Scone, NSW 2337, Australia",266500,NRMA Scone,"116 Liverpool Street, Scone, 2337",886745.3,14,operator_postcode_address,rule_based_accept,2023-06-08 13:24:00+10:00
1,ev_92a54038955adbf16c92062ab57a4b497be32cb889f...,"1 - 7 Ross St, Wilcannia NSW 2836, Australia",190678,Wilcannia [NRMA],"1-7 Ross St, Wilcannia, 2836",797549.3,13,operator_postcode_address,rule_based_accept,2022-05-04 13:08:00+10:00
2,ev_7cde3d4e59cd084712a54481b1dd4295c6fd2f3ab42...,"1 Little Walker St, Casino, NSW 2470, Australia",260392,Little Walker Street,"Little Walker Street, 2470",792873.6,14,operator_postcode_address,rule_based_accept,2023-04-23 20:48:00+10:00
3,ev_2f8f7072e8e1222bd4543430229589b8bf7b0d60ecf...,"Car park, 51 Evans St (Victoria Park), Inverel...",266480,NRMA Inverell,"59 Evans Street, Inverell , 2360",689090.3,12,operator_postcode_address,rule_based_accept,2023-06-06 22:14:00+10:00
4,ev_ee83de18c7561554fc8405e119c22a3010fa045294b...,"15c Mitchell St, Bourke, NSW 2840, Australia",190681,Bourke [NRMA],"15 Mitchell St, Bourke, 2840",651740.5,12,operator_postcode_address,rule_based_accept,2021-12-17 14:45:00+11:00
5,ev_4ce4a0703089f5a3d1c26a546e883295b73d1143549...,"82 Marsh St, Armidale, NSW 2350, Australia",170824,Armidale Visitors Centre,"82 Marsh Street, Armidale, 2350",584652.5,14,operator_postcode_address,rule_based_accept,2020-12-12 00:42:00+11:00
6,ev_fdd79f21a3b5b52d131cd997345814faa71f49945d2...,"Car park, Little Hoskins St, Temora, NSW 2666",190668,Temora Council Chambers,"Little Hoskins Street, Temora, 2666",540703.6,12,operator_postcode_address,rule_based_accept,2026-07-15 12:08:00+10:00
7,ev_0d5b32871d5bc6906b4287c8218180841e0651cf394...,"17 Stewart St, Wollongong NSW 2500, Australia",191177,Stewart Street Carpark [NRMA],"17 Stewart Street, Wollongong, 2500",517757.6,14,operator_postcode_address,rule_based_accept,2023-02-21 04:51:00+11:00
8,ev_9867e00742a07ac870361adfd69334d775ce64681c2...,"26 Neilly St, Walgett, NSW 2832, Australia",190683,Walgett [NRMA],"26 Neilly Street, Walgett, 2832",494538.8,14,operator_postcode_address,rule_based_accept,2021-12-17 15:13:00+11:00
9,ev_776ac39be06998cf2e32e1dc28783d7254274999809...,"10W Apsley St, Walcha NSW 2354, Australia",480135,NRMA EV Charger Walcha,"10W Apsley Street, Walcha, 2354",467746.2,13,operator_postcode_address,rule_based_accept,2026-03-15 19:37:00+11:00


,external_source,external_station_id,source_record_count
0,Open Charge Map API v3,272686,3
1,Open Charge Map API v3,266872,3
2,Open Charge Map API v3,190670,3
3,Open Charge Map API v3,272619,3
4,Open Charge Map API v3,272632,3
5,Open Charge Map API v3,274056,3
6,Open Charge Map API v3,266501,2
7,Open Charge Map API v3,312659,2
8,Open Charge Map API v3,272635,2
9,Open Charge Map API v3,190679,2


In [ ]:
with connect_database(PROJECT_ROOT) as con:
    # Geometries have different CRS: transform points explicitly before checking containment.
    display(con.execute("""
        SELECT count(*) AS confirmed_matches_rechecked,
               count(*) FILTER (WHERE ST_Within(
                   ST_Transform(c.geom, 'EPSG:4326', 'EPSG:7844', always_xy := true), r.geom
               )) AS matches_within_polygon
        FROM charger_records c JOIN charger_sa4 b USING(record_id)
        JOIN sa4_regions r USING(SA4_CODE26)
    """).df())
    display(con.execute("""
        SELECT count(*) AS dc_records,
               count(*) FILTER (WHERE has_augmentation) AS augmented_dc_records
        FROM charger_overview WHERE charger_type_standardized='DC'
    """).df())

,confirmed_matches_rechecked,matches_within_polygon
0,1957,1957


,dc_records,augmented_dc_records
0,433,216


## 5. Results and Data Quality Notes

Use the generated `data/final/quality_report.json` as the source of current results. For the supplied handoff files, the validated results are:

| Check | Result |
| --- | ---: |
| Base source records loaded | 1,958 |
| Standardized operators | 46 |
| SA4 categories retained | 108 |
| SA4 categories without geometry | 19 |
| Source records with an SA4 match | 1,957 |
| Unmatched source records retained | 1 |
| Differences in the independent spatial recheck | 0 |
| Invalid non-null region geometries | 0 |
| DC source records / rounded-coordinate locations | 433 / 430 |
| Augmented DC source records / locations | 216 / 215 |
| DC source-record coverage | 49.88% |
| DC location coverage using External Data Augmentation's definition | 50.00% |

The two coverage denominators must not be interchanged. Location grouping follows External Data Augmentation's six-decimal-place coordinate rule; it is not an independent physical-site identification process.

### Preserved data quality limitations

- The unmatched source record is retained. Its nearest candidate is stored separately and is not promoted to a confirmed region assignment.
- Seventeen accepted augmentation matches have distances greater than 3 km. Database and Integration preserves External Data Augmentation's decisions and exports these records for review rather than changing the upstream result.
- Charger coordinates retain Spatial Integration's provisional EPSG:4326 assumption. The spatial recheck validates agreement with Spatial Integration's implementation; it does not independently establish address-coordinate accuracy.
- Non-spatial SA4 categories remain in the region table but do not participate in geometric matching.

These limitations are recorded transparently and do not prevent reproduction of Database and Integration from the supplied inputs.

### Generated outputs

- `data/final/project.duckdb`: the populated relational and spatial database.
- `data/final/quality_report.json`: counts, validation results, software versions and input fingerprints.
- `data/final/sa4_summary.csv`: counts of source records by SA4.
- `data/final/augmentation_distance_review.csv`: accepted matches with distances greater than 3 km.

Input hashes identify the exact handoff version. Reproduction means rebuilding equivalent tables and validated results from those inputs; build timestamps may change, so the database file is not expected to be byte-for-byte identical.

